# 📖 Notebook 1: Producers and Consumers

Apache Kafka is the backbone behind real-time data at companies like LinkedIn, Netflix, and Uber. In this notebook, we'll learn the fundamentals by building a real-time World Cup event stream.

## Learning Objectives
- **What Kafka is** and why it exists (the World Cup analogy: events from games need to be processed in real-time)
- **How producers send messages** to topics
- **How consumers read messages** from topics
- **Understanding offsets** and message structure

## 🛠️ Setup

### 1. Start Kafka with Docker

```bash
cd 03-technologies/messaging/kafka
docker-compose up -d
```

This starts a Kafka broker (in KRaft mode — no Zookeeper needed) and a Kafka UI.

### 2. Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
If it doesn't appear, reload the VS Code window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from confluent_kafka.admin import AdminClient

KAFKA_CONFIG = {'bootstrap.servers': 'localhost:9092'}

try:
    admin = AdminClient(KAFKA_CONFIG)
    metadata = admin.list_topics(timeout=5)
    print(f"✅ Connected to Kafka! Broker has {len(metadata.topics)} topics")
except Exception as e:
    print(f"❌ Cannot connect to Kafka: {e}")
    print("   Run: cd 03-technologies/messaging/kafka && docker-compose up -d")

## 🤔 What is Kafka?

Imagine you're building a website that tracks **World Cup games in real-time**. Events happen constantly across multiple games:

- ⚽ A goal is scored in Brazil vs Germany
- 🟨 A yellow card is given in Argentina vs France
- 🔄 A substitution happens in Spain vs Portugal

Each of these events needs to reach **many different systems** — the live scoreboard, the notifications service, the statistics engine, the betting platform. How do you connect all of these reliably?

**Kafka is like a super-powered shared notebook** that sits in the middle:

```
                          ┌──────────────┐
  ⚽ Game Tracker  ──────▶│              │──────▶ 📊 Live Scoreboard
                          │              │
  📱 Mobile App   ──────▶│    KAFKA     │──────▶ 🔔 Notifications
                          │   (Topics)   │
  🎙️ Commentary   ──────▶│              │──────▶ 📈 Statistics
                          │              │
                          └──────────────┘
     PRODUCERS                                    CONSUMERS
   (write events)                              (read events)
```

### Why not just use a regular database?

| Feature | Database | Kafka |
|---------|----------|-------|
| Speed | Good for queries | Optimized for streaming |
| Consumers | One reader at a time | Many readers independently |
| Data flow | Pull (you ask for data) | Push/Pull (data flows to you) |
| Order | No guaranteed order | Ordered within partitions |
| Replay | Can't "re-read" easily | Can replay from any point |

Kafka's key insight: **messages are stored in an append-only log**. Producers write to the end, and each consumer tracks its own position. This means multiple consumers can read the same data at different speeds without interfering with each other.

## 📦 Message Structure

Every message in Kafka has 4 parts:

| Part | Required? | Description | Example |
|------|-----------|-------------|---------|
| **Key** | Optional | Used to decide which partition the message goes to. Messages with the same key always go to the same partition. | `"Brazil vs Germany"` |
| **Value** | Yes | The actual data/payload. Usually JSON. | `{"type": "goal", "player": "Neymar"}` |
| **Timestamp** | Auto | When the message was created. Kafka adds this automatically. | `1719432000000` |
| **Headers** | Optional | Extra metadata as key-value pairs. Like HTTP headers. | `{"source": "mobile-app"}` |

Think of it like an envelope:
- The **key** is who it's addressed to (determines routing)
- The **value** is the letter inside (the actual content)
- The **timestamp** is the postmark (when it was sent)
- The **headers** are notes on the envelope (extra info)

## 📁 Creating a Topic

Before we can send messages, we need a **topic**. Think of topics as **folders** or **categories** — they organize your messages by subject.

For example:
- `world-cup-events` — all game events (goals, cards, etc.)
- `user-signups` — new user registrations
- `order-payments` — payment transactions

Each topic is split into **partitions** (like splitting a folder into sub-folders). Partitions let Kafka handle more data by spreading it across multiple servers. We'll use 3 partitions for our topic.

In [ ]:
from confluent_kafka.admin import AdminClient, NewTopic

admin = AdminClient(KAFKA_CONFIG)

topic = NewTopic('world-cup-events', num_partitions=3, replication_factor=1)
futures = admin.create_topics([topic])

for topic_name, future in futures.items():
    try:
        future.result()
        print(f"✅ Topic '{topic_name}' created successfully!")
    except Exception as e:
        print(f"ℹ️ Topic '{topic_name}': {e}")

## 📨 Sending Messages (Producer)

A **producer** is any application that sends messages to Kafka. Here's how it works:

```
Your Code                    Kafka Broker
─────────                    ────────────
1. Create Producer    ──▶    
2. Call produce()     ──▶    Message queued internally
3. Call flush()       ──▶    Messages sent to broker
4. Callback fires     ◀──    Broker confirms delivery
```

**Important concepts:**
- `produce()` doesn't send immediately — it buffers messages for efficiency
- `flush()` waits until all buffered messages are actually sent
- The **delivery callback** tells you if each message was delivered successfully (and which partition/offset it landed on)

In [ ]:
from confluent_kafka import Producer
import json

producer = Producer(KAFKA_CONFIG)

def delivery_report(err, msg):
    if err:
        print(f"❌ Delivery failed: {err}")
    else:
        print(f"✅ Delivered to {msg.topic()} [partition {msg.partition()}] @ offset {msg.offset()}")

events = [
    {"match": "Brazil vs Germany", "type": "goal", "player": "Neymar", "minute": 23},
    {"match": "Brazil vs Germany", "type": "yellow_card", "player": "Müller", "minute": 35},
    {"match": "Argentina vs France", "type": "goal", "player": "Messi", "minute": 10},
    {"match": "Brazil vs Germany", "type": "substitution", "player_out": "Silva", "player_in": "Jesus", "minute": 60},
    {"match": "Argentina vs France", "type": "goal", "player": "Mbappé", "minute": 78},
]

for event in events:
    producer.produce(
        topic='world-cup-events',
        value=json.dumps(event).encode('utf-8'),
        callback=delivery_report,
    )

producer.flush()
print(f"\n📨 Sent {len(events)} events!")

## 📥 Reading Messages (Consumer)

A **consumer** reads messages from one or more topics. Here's what you need to know:

- **`group.id`** — Consumers belong to a "consumer group". Kafka tracks each group's progress separately. Think of it like a bookmark — each reading group has its own bookmark.
- **`auto.offset.reset = 'earliest'`** — If this consumer group has never read this topic before, start from the **very first message**. The alternative is `'latest'` which only reads new messages.
- **`poll()`** — Asks Kafka "do you have any new messages for me?" and waits up to the timeout.

```
Consumer                        Kafka Broker
────────                        ────────────
1. Subscribe to topic   ──▶    
2. Poll for messages    ──▶    "Here are messages 0, 1, 2"
3. Process messages     ◀──    
4. Poll again           ──▶    "Here are messages 3, 4"
5. Close consumer       ──▶    Offset committed
```

In [ ]:
from confluent_kafka import Consumer

consumer_config = {
    **KAFKA_CONFIG,
    'group.id': 'notebook-1-basic',
    'auto.offset.reset': 'earliest',
}

consumer = Consumer(consumer_config)
consumer.subscribe(['world-cup-events'])

print("📥 Reading messages from 'world-cup-events'...")
print("=" * 60)

msg_count = 0
empty_polls = 0

while empty_polls < 3:
    msg = consumer.poll(timeout=2.0)
    if msg is None:
        empty_polls += 1
        continue
    if msg.error():
        print(f"❌ Error: {msg.error()}")
        continue

    empty_polls = 0
    msg_count += 1
    value = json.loads(msg.value().decode('utf-8'))
    print(f"  Message #{msg_count}: partition={msg.partition()}, offset={msg.offset()}")
    print(f"    {value}")
    print()

consumer.close()
print(f"📊 Total messages read: {msg_count}")

## 🔢 Understanding Offsets

**Offsets are like page numbers in a book.** Each message within a partition gets a sequential number starting from 0. When a consumer reads messages, Kafka remembers which "page" (offset) the consumer has read up to.

```
Partition 0:  ┌─────┬─────┬─────┬─────┬─────┐
              │  0  │  1  │  2  │  3  │  4  │  ──▶ new messages go here
              └─────┴─────┴─────┴─────┴─────┘
                          ▲
                          │
                  Consumer's current
                    offset = 2
                ("I've read up to here")
```

### Why offsets matter:
- If a consumer **crashes and restarts**, it picks up from its last committed offset — no messages lost!
- Different consumer groups have **independent offsets** — they each have their own bookmark
- You can even **reset offsets** to replay old messages (great for debugging)

Let's see this in action...

In [ ]:
# First read: consume only 2 messages, then stop
config = {**KAFKA_CONFIG, 'group.id': 'offset-demo', 'auto.offset.reset': 'earliest', 'enable.auto.commit': True}
c = Consumer(config)
c.subscribe(['world-cup-events'])

print("📖 First read: consuming 2 messages then stopping...")
count = 0
while count < 2:
    msg = c.poll(timeout=5.0)
    if msg is None:
        continue
    if msg.error():
        continue
    count += 1
    value = json.loads(msg.value().decode('utf-8'))
    print(f"  Read #{count}: offset={msg.offset()} → {value.get('type', '')} by {value.get('player', 'N/A')}")

c.close()
print("⏸️ Consumer stopped. Offset committed.\n")

import time
time.sleep(1)

# Second read: resume from where we left off
c2 = Consumer(config)
c2.subscribe(['world-cup-events'])

print("📖 Second read: resuming from last committed offset...")
count = 0
empty = 0
while empty < 3:
    msg = c2.poll(timeout=2.0)
    if msg is None:
        empty += 1
        continue
    if msg.error():
        continue
    empty = 0
    count += 1
    value = json.loads(msg.value().decode('utf-8'))
    print(f"  Read #{count}: offset={msg.offset()} → {value.get('type', '')} by {value.get('player', 'N/A')}")

c2.close()
print(f"\n✅ Resumed and read {count} remaining messages!")
print("💡 Kafka remembered where we left off!")

## 🔑 Using Message Keys

Remember how topics are split into **partitions**? When you send a message, Kafka needs to decide **which partition** it goes to.

- **No key** → Kafka picks a partition using round-robin (spreads evenly)
- **With a key** → Kafka hashes the key to pick a partition (same key = same partition, always)

```
Key: "Brazil vs Germany"  ──hash──▶ Partition 1
Key: "Argentina vs France" ──hash──▶ Partition 0
Key: "Brazil vs Germany"  ──hash──▶ Partition 1  (same key = same partition!)
```

### Why use keys?
Messages in the **same partition are always read in order**. So if you use the match name as a key, all events for one game land on the same partition, and the consumer processes them in the correct chronological order.

Without keys, a "goal" event might arrive before the "kickoff" event if they land on different partitions!

In [ ]:
producer = Producer(KAFKA_CONFIG)

events_with_keys = [
    ("Brazil vs Germany", {"type": "kickoff", "minute": 0}),
    ("Argentina vs France", {"type": "kickoff", "minute": 0}),
    ("Brazil vs Germany", {"type": "goal", "player": "Neymar", "minute": 12}),
    ("Argentina vs France", {"type": "corner", "minute": 15}),
    ("Brazil vs Germany", {"type": "halftime", "minute": 45}),
    ("Argentina vs France", {"type": "goal", "player": "Messi", "minute": 50}),
]

print("📨 Sending events with keys (match name as key)...")
for key, event in events_with_keys:
    event['match'] = key
    producer.produce(
        topic='world-cup-events',
        key=key.encode('utf-8'),
        value=json.dumps(event).encode('utf-8'),
        callback=delivery_report,
    )

producer.flush()
print(f"\n💡 Events for the same match always land on the same partition!")
print("   This ensures per-match ordering is preserved.")

## 🖥️ Kafka UI: See It Visually

Open **http://localhost:8080** in your browser to see the Kafka UI.

Try these:
1. Click on **Topics** → **world-cup-events** to see the messages you just sent
2. Check which **partition** each message landed on
3. Notice how messages with the same key are on the same partition
4. Click on **Consumers** to see your consumer groups and their offsets

## 📝 Key Takeaways

- **Kafka stores messages in topics** (categories) split into **partitions** (for parallel processing and scale)
- **Producers send messages**, consumers read them — they are completely decoupled
- Each message has a **key** (routing), **value** (data), **timestamp** (when), and **headers** (metadata)
- **Offsets track consumer position** — Kafka remembers where you left off, even across restarts
- **Message keys control partitioning** — same key = same partition
- **Same partition = ordered processing** — critical for related data that must stay in sequence

## ➡️ What's Next

In the next notebook, we'll explore **partitioning strategies and consumer groups** — how Kafka distributes work across multiple consumers and how you can scale your processing.